In [ ]:
import ROOT
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt

# === Configuration ===
year = "2017"
bgr = "ttcr1"
folder = f"/eos/user/j/jreyesve/WINDOWS/Desktop/2025/Noviembre/Plotter/root_files/tt/2017/root"

# === Dictionary to hold all data in memory ===
tables = {}

# === List ROOT files ===
root_files = [f for f in os.listdir(folder) if f.endswith('.root') and not f.startswith('.')]
print(f"Processing {len(root_files)} ROOT files...")

for filename in root_files:
    print(f"\nProcessing: {filename}")
    file_path = f"{folder}/{filename}"

    try:
        file = ROOT.TFile.Open(file_path)
        if not file or file.IsZombie():
            print(f"  ✗ Skipping invalid file: {filename}")
            continue

        # --- Collect histograms ---
        histograms = []
        for key in file.GetListOfKeys():
            obj = key.ReadObj()
            if obj and obj.InheritsFrom("TH1"):
                histograms.append(obj)

        if not histograms:
            file.Close()
            print(f"  ⚠️ No histograms found in {filename}")
            continue

        # --- Build a table (DataFrame) with binning information ---
        first_hist = histograms[0]
        nbins = first_hist.GetNbinsX()
        bin_data = {
            "bin_number": np.arange(1, nbins + 1),
            "low_edge": [first_hist.GetBinLowEdge(i) for i in range(1, nbins + 1)],
            "center": [first_hist.GetBinCenter(i) for i in range(1, nbins + 1)],
            "high_edge": [first_hist.GetBinLowEdge(i) + first_hist.GetBinWidth(i) for i in range(1, nbins + 1)],
            "width": [first_hist.GetBinWidth(i) for i in range(1, nbins + 1)],
        }

        # Add each histogram as a column
        for hist in histograms:
            contents = [hist.GetBinContent(i) for i in range(1, nbins + 1)]
            bin_data[hist.GetName()] = contents

        df = pd.DataFrame(bin_data)

        # Store the DataFrame in the dictionary
        tables[filename] = df

        print(f"  ✓ Loaded {len(histograms)} histograms with {nbins} bins")

        file.Close()

    except Exception as e:
        print(f"  ✗ Error processing {filename}: {str(e)}")

print("\n✅ All ROOT files loaded in memory.")
print("You can access them with:")
print("  tables['<filename>.root']")

# === Optional: helper function to visualize a histogram ===
def show_histogram(file_name, hist_name):
    """
    Plot a histogram from the in-memory table.

    Args:
        file_name (str): ROOT filename key in `tables`.
        hist_name (str): Column name of the histogram to plot.
    """
    if file_name not in tables:
        print(f"⚠️ File '{file_name}' not found in tables.")
        return
    df = tables[file_name]

    if hist_name not in df.columns:
        print(f"⚠️ Histogram '{hist_name}' not found in {file_name}.")
        print("Available histograms:", [c for c in df.columns if c not in ['bin_number','low_edge','center','high_edge','width']])
        return

    plt.figure(figsize=(8, 5))
    plt.bar(df["center"], df[hist_name], width=df["width"], align="center", alpha=0.7)
    plt.xlabel("X-axis (variable)")
    plt.ylabel("Entries")
    plt.title(f"{hist_name} — {file_name}")
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:

from IPython.display import display  # For pretty DataFrame display in notebooks

for root_name in list(tables.keys()):
    df = tables[root_name]
    
    print("==================================================================")
    print(f"📁 ROOT file: {root_name}")
    
    # --- List all available histograms in the file ---
    all_hists = [c for c in df.columns 
                 if c not in ['bin_number', 'low_edge', 'center', 'high_edge', 'width']]
    

    # --- Find the histogram that ends with 'nom' ---
    hist_candidates = [c for c in all_hists if c.endswith("nom")]
    
    if not hist_candidates:
        print(f"⚠️ No histogram ending with 'nom' found in {root_name}.\n")
        continue
    
    hist_name = hist_candidates[0]
    
    # --- Extract histogram info ---
    nbins = len(df)
    xmin = df["low_edge"].min()
    xmax = df["high_edge"].max()
    
    print(f"\n📊 Selected histogram (ends with 'nom'): {hist_name}")
    print(f"   • Number of bins: {nbins}")
    print(f"   • Range: [{xmin:.2f}, {xmax:.2f}]")
    
    total_events = df[hist_name].sum()
    print(f"   • Total events (nom): {total_events}\n")
    
    
    # ============================================================
    #       NEW SECTION: Integrals of all CMS_* histograms
    # ============================================================
    cms_hists = [h for h in all_hists if h.startswith("CMS_")]
    
    if cms_hists:
        print("🧮 Integrales de histogramas CMS_*:")
        
        # Agrupar Up/Down
        cms_groups = {}
        for h in cms_hists:
            if h.endswith("_Up"):
                base = h.replace("_Up", "")
                cms_groups.setdefault(base, {})["Up"] = h
            elif h.endswith("_Down"):
                base = h.replace("_Down", "")
                cms_groups.setdefault(base, {})["Down"] = h
        
        # Imprimir integrales ordenadas
        for base_name, entries in sorted(cms_groups.items()):
            up_int = df[entries["Up"]].sum() if "Up" in entries else None
            down_int = df[entries["Down"]].sum() if "Down" in entries else None
            
            print(f"   • {base_name}:")
            if up_int is not None:
                print(f"       ↑ Up:   {up_int}")
            if down_int is not None:
                print(f"       ↓ Down: {down_int}")
    
    print("\n")
    
    # --- Add a total row to display table ---
    total_row = {
        "low_edge": "Total",
        "high_edge": "",
        hist_name: total_events
    }
    df_with_total = pd.concat(
        [df[["low_edge", "high_edge", hist_name]], pd.DataFrame([total_row])],
        ignore_index=True
    )

    display(df_with_total)

In [ ]:
from IPython.display import display  # For pretty DataFrame display in notebooks

for root_name in list(tables.keys()):
    df = tables[root_name]
    
    print("==================================================================")
    print(f"📁 ROOT file: {root_name}")
    
    # --- List all available histograms in the file ---
    all_hists = [c for c in df.columns 
                 if c not in ['bin_number', 'low_edge', 'center', 'high_edge', 'width']]
    
    print("📜 Available histograms:")
    for h in all_hists:
        print(f"   • {h}")
    
    # --- Find the histogram that ends with 'nom' ---
    hist_candidates = [c for c in all_hists if c.endswith("nom")]
    
    if not hist_candidates:
        print(f"⚠️ No histogram ending with 'nom' found in {root_name}.\n")
        continue
    
    # Use the first histogram found that matches the 'nom' pattern
    hist_name = hist_candidates[0]
    
    # --- Extract histogram info ---
    nbins = len(df)
    xmin = df["low_edge"].min()
    xmax = df["high_edge"].max()
    
    print(f"\n📊 Selected histogram (ends with 'nom'): {hist_name}")
    print(f"   • Number of bins: {nbins}")
    print(f"   • Range: [{xmin:.2f}, {xmax:.2f}]\n")
    
    # --- Add a total row at the bottom ---
    total_value = df[hist_name].sum()
    total_row = {
        "low_edge": "Total",
        "high_edge": "",
        hist_name: total_value
    }
    df_with_total = pd.concat([df[["low_edge", "high_edge", hist_name]], pd.DataFrame([total_row])], ignore_index=True)
    
    # --- Display the histogram data as a pandas table with total row ---
    display(df_with_total)

